In [34]:
import math

def convert_entry(d):
    out = {}

    out["name"] = d["name"]
    out["family"] = d["family"]
    out["subfamily"] = d["type"]
    out["size"] = d["size"]

    out["parameters"] = {}

    # helper to create parameter blocks
    def set_param(key, value, units):
        out["parameters"][key] = {"value": value, "units": units}

    # base values
    set_param("maxSpeed", d["maxSpeed"], "rpm")

    set_param("inertia1", d["inertia_1"], "kgm^2")
    set_param("inertia2", d["inertia_2"], "kgm^2")

    set_param("mass1", d["mass_1"], "kg")
    set_param("mass2", d["mass_2"], "kg")

    set_param("shoreHardness", d["shoreHardness"], "")

    # stiffness (kNm/rad → Nm/rad)
    set_param("torsionalStiffness", d["torsionalStiffness"] * 1000, "Nm/rad")

    # angular stiffness kNm/deg → Nm/rad
    ang_stiff = round((d["angularStiffness"] * 1000) / math.radians(1), 4)
    set_param("angularStiffness", ang_stiff, "Nm/rad")
    
    # stiffness kN/mm → N/m (× 1e6)
    set_param("axialStiffness", d["axialStiffness"] * 1e6, "N/m")
    set_param("radialStiffness", d["radialStiffness"] * 1e6, "N/m")

    # torques (kNm → Nm)
    set_param("nominalTorque", d["nominalTorque"] * 1000, "Nm")
    set_param("maxTorque", d["maxTorque"] * 1000, "Nm")
    set_param("continuousVibratoryTorque", d["continuousVibratoryTorque"] * 1000, "Nm")

    # power kW → W
    set_param("permissiblePowerLoss", d["permissiblePowerLoss"] * 1000, "W")

    # displacement mm → m
    set_param("permissibleAxialDisplacement", d["permissibleAxialDisplacement"] / 1000, "m")
    set_param("permissibleRadialDisplacement", d["permissibleRadialDisplacement"] / 1000, "m")

    # angle deg → rad
    ang_disp = round((d["permissibleAngularDisplacement"] * 1) * math.radians(1), 4)
    set_param("permissibleAngularDisplacement", ang_disp, "rad")

    set_param("relativeDamping", d["relativeDamping"], "")

    # torsional damping
    I1 = d["inertia_1"]
    I2 = d["inertia_2"]
    Ieq = (I1 * I2) / (I1 + I2)
    k = out["parameters"]["torsionalStiffness"]["value"]
    rel = d["relativeDamping"]
    damping = round(rel * math.sqrt(Ieq * k), 4)
    set_param("torsionalDamping", damping, "Nms/rad")

    return out


In [35]:
from pathlib import Path
import json

# Load input file
path = Path("archive/all_coupllings.json")


def gather_and_build_couplings(path):
    with open(path, "r") as f:
        families = json.load(f)

    couplings = []
    recompiled_couplings = []

    for family in families:
        family_base = {k: v for k, v in family.items() if k != "options"}
        family_name = family_base.get("name", "")
        options = family.get("options", [])

        for option in options:
            option_name = option.get("name", "")
            full_name = f"{family_name}_{option_name}"

            merged = family_base.copy()
            merged.update(option)
            merged["name"] = full_name

            couplings.append(merged)

    for coupling in couplings:
        recompiled_coupling = convert_entry(coupling)
        recompiled_couplings.append(recompiled_coupling)

    # Save output
    out_path = Path("final_recompiled_couplings.json")
    with open(out_path, "w") as f:
        json.dump(recompiled_couplings, f, indent=2)


In [36]:
gather_and_build_couplings(path)

In [28]:
inn = {
    "name": "Centaflex_Size1_Type_0_shore50",
    "family": "CENTAFLEX-A",
    "size": 1,
    "maxSpeed": 10000.0, # rpm
    "type": "Type 0",
    "inertia_1": 2e-05, # kgm2
    "mass_1": 0.04, #kg
    "inertia_2": 2e-05, # kgm2
    "mass_2": 0.05, #kg
    "selectedOption": "shore50",
    "shoreHardness": 50,
    "torsionalStiffness": 0.09, #Knm/rad
    "relativeDamping": 0.6,
    "nominalTorque": 0.01, #kNm
    "maxTorque": 0.025, #kNm
    "continuousVibratoryTorque": 0.005, #kNm
    "permissiblePowerLoss": 6.0, #kW
    "permissibleAxialDisplacement": 2.0, #mm
    "axialStiffness": 0.0266, #kN/mm
    "permissibleRadialDisplacement": 1.5, #mm 
    "radialStiffness": 0.105, #kN/mm
    "permissibleAngularDisplacement": 3.0, #deg
    "angularStiffness": 0.00021 #kNm/deg
}

outi ={
    "name": "Centaflex_Size1_Type_0_shore50",
    "family": "CENTAFLEX-A",
    "size": 1,
    "maxSpeed": 10000.0, # rpm
    "subfamily": "Type 0",
    "inertia_1": 2e-05, # kgm2
    "mass_1": 0.04, #kg
    "inertia_2": 2e-05, # kgm2
    "mass_2": 0.05, #kg
    "shoreHardness": 50,
    "torsionalStiffness": 90, # convert to Nm/rad
    "relativeDamping": 0.6,
    "torsionalDamping": 4, # get from I, relative damping and k as rel. damping * sqrt(I*k) where I is I1*I2 /I1+I2
    "nominalTorque": 10, # convert to Nm
    "maxTorque": 25, ## convert to Nm
    "continuousVibratoryTorque": 5, #Nm
    "permissiblePowerLoss": 6000, #W
    "permissibleAxialDisplacement": 2e-3, #m
    "axialStiffness": 2.66e4, #kN/mm 106
    "permissibleRadialDisplacement": 1.5e-3, #m 
    "radialStiffness": 1.05e4, # Nm/m
    "permissibleAngularDisplacement": 3.0, # convert to rad
    "angularStiffness": 0.00021 # convert to Nm/rad
}

In [33]:
outt = convert_entry(inn)
import json

print(json.dumps(outt, indent=4))


{
    "name": "Centaflex_Size1_Type_0_shore50",
    "family": "CENTAFLEX-A",
    "subfamily": "Type 0",
    "size": 1,
    "parameters": {
        "maxSpeed": {
            "value": 10000.0,
            "units": "rpm"
        },
        "inertia1": {
            "value": 2e-05,
            "units": "kgm^2"
        },
        "inertia2": {
            "value": 2e-05,
            "units": "kgm^2"
        },
        "mass1": {
            "value": 0.04,
            "units": "kg"
        },
        "mass2": {
            "value": 0.05,
            "units": "kg"
        },
        "shoreHardness": {
            "value": 50,
            "units": ""
        },
        "torsionalStiffness": {
            "value": 90.0,
            "units": "Nm/rad"
        },
        "angularStiffness": {
            "value": 12.0321,
            "units": "Nm/rad"
        },
        "axialStiffness": {
            "value": 26600.0,
            "units": "N/m"
        },
        "radialStiffness": {
            